In [ ]:
#!python -m pip install --upgrade pip
#%pip install pandas matplotlib seaborn scikit-learn openpyxl tensorflow xgboost aif360
#%pip install "aif360[Reductions, inFairness]"

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from collections import Counter
from scipy.stats import chi2_contingency, fisher_exact

from fairlearn.metrics import MetricFrame
from fairlearn.metrics import demographic_parity_ratio, equalized_odds_ratio 
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference, selection_rate, false_positive_rate, false_negative_rate, count
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

random_seed = 15

In [ ]:
PATH = 'C:/Users/andre/Desktop/ProjectWork_AEQUITAS_AKKODIS/'

with open(PATH + 'data/predictions.json', 'r') as f:
    data = json.load(f)
predictions_df = pd.DataFrame(data['predictions'])
print(predictions_df.shape)
y_test = pd.Series(data['reference'])
print(y_test.shape)
s_test = pd.Series(data['sensitive'])
print(s_test.shape)
sensitive_feature = data['sensitive_name']
predictions_df.head(50)

In [ ]:
with open(PATH + 'data/encoding_mappings.json', 'r') as f:
    encoding_mappings = json.load(f)
pprint(encoding_mappings)

## Fairness Metrics

In [ ]:
metrics = []
for name in predictions_df.columns:
    y_pred = predictions_df[name]
    accuracy = round(accuracy_score(y_test, y_pred), 3)
    precision = round(precision_score(y_test, y_pred), 3)
    recall = round(recall_score(y_test, y_pred), 3)
    f1 = round(f1_score(y_test, y_pred), 3)
    roc_auc = round(roc_auc_score(y_test, y_pred), 3)

    metrics.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC AUC': roc_auc
    })
metrics = pd.DataFrame(metrics)

In [ ]:
def compute_fairness_metrics(y_true, y_pred, s_test, label=None):
    mf = MetricFrame(
        metrics={
            'selection_rate': selection_rate,
            'fpr': false_positive_rate,
            'fnr': false_negative_rate,
            'count': count
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=s_test
    )

    dp_diff = demographic_parity_difference(y_true, y_pred, sensitive_features=s_test)
    eo_diff = equalized_odds_difference(y_true, y_pred, sensitive_features=s_test)

    dp = demographic_parity_ratio(y_true, y_pred, sensitive_features=s_test)
    eo = equalized_odds_ratio(y_true, y_pred, sensitive_features=s_test)

    if label:
        print(f"=== {label} ===")

    print("By group:")
    print(mf.by_group)
    print()
    print("Overall (selection_rate, fpr, fnr, count):")
    print(mf.overall)
    print()
    print(f"Demographic parity difference: {dp_diff:.4f}")
    print(f"Equalized odds difference:     {eo_diff:.4f}\n")
    print()
    print(f"Demographic parity ratio: {dp:.4f}")
    print(f"Equalized odds ratio:     {eo:.4f}\n")

    return mf
for name in predictions_df.columns:
    compute_fairness_metrics(y_test, predictions_df[name], s_test, label=name)

#### **3.1 Demographic Parity**

In [ ]:
tolerance = 0.15
significance_level = 0.1

In [ ]:
def calculate_demographic_parity(predictions, sensitive_attribute, name, significance_level, tolerance, activate_check=False):

    df = pd.DataFrame({
        'predictions': predictions,
        'sensitive_attribute': sensitive_attribute
    })
    prop = df.groupby('sensitive_attribute')['predictions'].mean()
    
    if activate_check:
        print(f"=== {name} ===")
        print(f"{prop}")

    if prop.shape[0] == 2:
        diff = prop.max() - prop.min()
        if activate_check:
            print(f"Two groups: |Δ| = {diff:.4f}, tol = {tolerance}")
        return 'T' if diff <= tolerance else False
    
    contingency_table = pd.crosstab(df['predictions'], df['sensitive_attribute'])
    chi2, p, dof, expected = chi2_contingency(contingency_table, correction=False)
    
    if contingency_table.shape == (2, 2) and (expected < 5).any():
        _, p = fisher_exact(contingency_table)
        if activate_check:
            print(f"Fisher’s exact test fallback for {name}")
    elif contingency_table.shape != (2, 2) and (expected < 5).any():
        if activate_check:
            print(f"Sparse contingency for {name}")
        
    return 'T' if p > significance_level else False    

table = []
for name in predictions_df.columns:
    result = calculate_demographic_parity(predictions_df[name], s_test, name, significance_level, tolerance, activate_check=True)
    table.append(result)
sf_df = pd.DataFrame(table, index = predictions_df.columns, columns=[sensitive_feature])

#### **3.2 Equalized odds**

In [ ]:
tolerance = 0.15
significance_level = 0.1

In [ ]:
def calculate_equalized_odds(predictions, true_labels, sensitive_attribute, name, tolerance, activate_check=False):
    df = pd.DataFrame({
        'predictions': predictions,
        'true_labels': true_labels,
        'sensitive_attribute': sensitive_attribute
    })
    tprs, fprs = [], []
    for _, group_df in df.groupby('sensitive_attribute'):
        tn, fp, fn, tp = confusion_matrix(group_df['true_labels'], group_df['predictions'], labels=[0, 1]).ravel()
        tprs.append(tp / (tp + fn) if tp + fn != 0 else 0)
        fprs.append(fp / (fp + tn) if fp + tn != 0 else 0)

    max_tpr_diff = max(tprs) - min(tprs)
    max_fpr_diff = max(fprs) - min(fprs)

    if activate_check:
            print(f"=== {name} ===")
            print(f"Max FPR difference: {max_fpr_diff}")
            print(f"Max TPR difference: {max_tpr_diff}")

    return 'T' if (max_tpr_diff <= 2 * tolerance and max_fpr_diff <= 2 * tolerance) else False


table = []
for name in predictions_df.columns:
    result = calculate_equalized_odds(predictions_df[name], y_test, s_test, name, tolerance, activate_check=True)
    table.append(result)
sf_df = pd.DataFrame(table, index = predictions_df.columns, columns=[sensitive_feature])